# Engagement Classification — MLP Training Notebook

This notebook trains a **Multi-Layer Perceptron (MLP)** classifier to predict student engagement states from facial landmark features.

**Classes:** `0 = ATTENTIVE` | `1 = SLEEPY` | `2 = DISTRACTED`

**Input features (9 total):**
| Feature | Description |
|---|---|
| `ear_left`, `ear_right`, `ear_avg` | Eye Aspect Ratio — measures eye openness |
| `mar` | Mouth Aspect Ratio — detects yawning |
| `pitch`, `yaw`, `roll` | Head orientation in degrees via solvePnP |
| `perclos` | Fraction of frames with eyes closed over a 3-second rolling window |
| `blink_rate` | Blinks per minute over a 60-second rolling window |

**Pipeline:**
```
labeled_features.csv → EDA → Preprocessing → MLP Training → Evaluation → Save Model
```

## Step 1: Install Dependencies

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn joblib

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3: Configuration

In [ ]:
from pathlib import Path

DATA_PATH  = Path('/content/drive/MyDrive/COS40007_Project/landmark_pipeline/labeled_features.csv')
MODELS_DIR = Path('/content/drive/MyDrive/COS40007_Project/landmark_pipeline/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = [
    'ear_left', 'ear_right', 'ear_avg',
    'mar',
    'pitch', 'yaw', 'roll',
    'perclos',
    'blink_rate',
]
LABEL_NAMES = {0: 'ATTENTIVE', 1: 'SLEEPY', 2: 'DISTRACTED'}

print(f'Data path : {DATA_PATH}')
print(f'Models dir: {MODELS_DIR}')

## Step 4: Load Dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(DATA_PATH)

print(f'Total samples: {len(df)}')
print(f'Columns: {list(df.columns)}')
print()
print('Class distribution:')
print(df['label'].map(LABEL_NAMES).value_counts().to_string())
print()
df.head()

## Step 5: Exploratory Data Analysis (EDA)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df['label_name'] = df['label'].map(LABEL_NAMES)

# Class distribution bar chart
plt.figure(figsize=(6, 4))
counts = df['label_name'].value_counts()
colors = ['#2ecc71', '#e67e22', '#e74c3c']
plt.bar(counts.index, counts.values, color=colors)
plt.title('Class Distribution')
plt.xlabel('Engagement State')
plt.ylabel('Sample Count')
for i, v in enumerate(counts.values):
    plt.text(i, v + 1, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print('Target: 200+ samples per class')
for name, count in counts.items():
    status = '✓' if count >= 200 else '✗ needs more data'
    print(f'  {name}: {count} {status}')

In [ ]:
# Feature distributions per class
key_features = ['ear_avg', 'mar', 'yaw', 'pitch', 'perclos', 'blink_rate']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    for label_id, label_name in LABEL_NAMES.items():
        subset = df[df['label'] == label_id][feat]
        axes[i].hist(subset, bins=30, alpha=0.6, label=label_name)
    axes[i].set_title(feat)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')
    axes[i].legend()

plt.suptitle('Feature Distributions per Engagement Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature statistics per class
df.groupby('label_name')[FEATURE_COLS].mean().round(3)

## Step 6: Preprocessing

- **Stratified 80/20 train/test split** — preserves class proportions in both sets
- **StandardScaler** — normalises each feature to zero mean and unit variance so the MLP is not sensitive to feature scale differences

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[FEATURE_COLS].values.astype(np.float32)
y = df['label'].values.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Feature count    : {X_train.shape[1]}')

## Step 7: Train MLP Classifier

**Architecture:** Input(9) → Hidden(64) → Hidden(32) → Output(3)  
**Activation:** ReLU  
**Optimiser:** Adam (default in sklearn MLPClassifier)

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    max_iter=500,
    random_state=42,
    verbose=False,
)

mlp.fit(X_train_s, y_train)

print(f'Training complete after {mlp.n_iter_} iterations')
print(f'Final training loss: {mlp.loss_:.4f}')

## Step 8: Training Loss Curve

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(mlp.loss_curve_, color='steelblue', linewidth=2)
plt.title('MLP Training Loss Curve')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 9: Evaluate

- **Precision** — of all samples predicted as class X, how many were actually X
- **Recall** — of all actual class X samples, how many did the model find
- **F1** — harmonic mean of precision and recall

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = mlp.predict(X_test_s)

print('Classification Report')
print('=' * 50)
print(classification_report(y_test, y_pred, target_names=list(LABEL_NAMES.values())))

In [ ]:
# Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred)
label_list = list(LABEL_NAMES.values())

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_list,
    yticklabels=label_list,
)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

## Step 10: Save Model

Saves `mlp_model.pkl` and `scaler.pkl` to Google Drive. These files are loaded by `pipeline.py` at runtime when running with `--mlp`.

In [ ]:
import joblib

model_path  = MODELS_DIR / 'mlp_model.pkl'
scaler_path = MODELS_DIR / 'scaler.pkl'

joblib.dump(mlp,    model_path)
joblib.dump(scaler, scaler_path)

print(f'Model saved  : {model_path}')
print(f'Scaler saved : {scaler_path}')
print()
print('To use in pipeline.py, copy both .pkl files into the local models/ folder')
print('then run: python src/classifiers/landmark_pipeline/pipeline.py --mlp')